# ਪਾਠ 08 - ਮਲਟੀ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅਪ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਕਿਉਂ ਮਲਟੀ-ਏਜੰਟ ਸਿਸਟਮ?

ਅਸਲ ਦੁਨੀਆ ਦੇ ਕੰਮ ਜਿਵੇਂ ਯਾਤਰਾ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿਚ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀ ਮਹਾਰਤ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ — ਲੋਜਿਸਟਿਕਸ, ਸਥਾਨਕ ਜਾਣਕਾਰੀਆਂ, ਬਜਟ ਬਣਾਉਣਾ, ਅਤੇ ਹੋਰ. ਇੱਕ ਹੀ ਏਜੰਟ ਜੋ ਸਾਰਾ ਕੰਮ ਸੰਭਾਲਣ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ, ਉਹ ਜਲਦੀ ਹੀ ਸੰਭਾਲਣ ਯੋਗ ਨਹੀਂ ਰਹਿੰਦਾ.

ਮਲਟੀ-ਏਜੰਟ ਸਿਸਟਮ ਇਸ ਨੂੰ **ਮਾਹਰਤਾ** ਦੇ ਜਰੀਏ ਹੱਲ ਕਰਦੇ ਹਨ: ਹਰ ਏਜੰਟ ਇੱਕ ਖੇਤਰ ਵਿਚ ਧਿਆਨ ਕੇਂਦ੍ਰਿਤ ਕਰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਆਮ ਤਜਰਬਾਕਾਰ ਨਾਲੋਂ ਵਧੀਆ ਨਤੀਜੇ ਮਿਲਦੇ ਹਨ. ਇਹਨਾਂ ਨਾਲ **ਪੈਮਾਨੇਯੋਗਤਾ** ਵੀ ਸੁਧਰਦੀ ਹੈ — ਤੁਸੀਂ ਨਵੇਂ ਏਜੰਟ (ਜਿਵੇਂ ਕਿ ਉਡਾਣ ਮਾਹਿਰ, ਰੈਸਟੋਰੈਂਟ ਸਮੀਖਿਆਕਾਰ) ਜੋੜ ਸਕਦੇ ਹੋ ਬਿਨਾਂ ਮੌਜੂਦਾ ਵਰਕਫਲੋ ਨੂੰ ਮੁੜ ਲਿਖਣ ਦੇ. ਏਜੰਟ ਇੱਕ ਸੰਗਠਿਤ ਪਾਈਪਲਾਈਨ ਰਾਹੀਂ ਇਕੱਠੇ ਕੰਮ ਕਰਦੇ ਹਨ, ਇੱਕ ਤੋਂ ਦੂਜੇ ਨੂੰ ਪ੍ਰਸੰਗ ਵਾਹਤ ਕਰਦੇ ਹੋਏ.


## ਵਿਸ਼ੇਸ਼ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਬਣਾਉਣਾ

`WorkflowBuilder` ਤੁਹਾਨੂੰ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਦਿਸ਼ਾਦਰਸ਼ਿਤ ਗ੍ਰਾਫ ਵਿੱਚ ਜੋੜਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ। ਇੱਥੇ ਅਸੀਂ ਇੱਕ ਸੌਖਾ ਦੋ-ਕਦਮ ਵਾਲਾ ਪਾਈਪਲਾਈਨ ਬਣਾਉਂਦੇ ਹਾਂ: **TravelPlanner** ਯਾਤਰਾ ਦਾ ਰੂਟ ਤਿਆਰ ਕਰਦਾ ਹੈ, ਫਿਰ **TravelConcierge** ਉਸ ਦੀ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ ਅਤੇ ਇਸ ਨੂੰ ਸੁਧਾਰਦਾ ਹੈ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਵਰਕਫਲੋ ਵਿੱਚ ਹੋਰ ਏਜੰਟ ਸ਼ਾਮਲ ਕਰਨਾ

ਮਲਟੀ-ਏਜੰਟ ਪੈਟਰਨ ਦਾ ਸਭ ਤੋਂ ਵੱਡਾ ਫਾਇਦਾ ਇਹ ਹੈ ਕਿ ਇਸਨੂੰ ਵਧਾਉਣਾ ਕਿੰਨਾ ਆਸਾਨ ਹੈ। ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ **BudgetReviewer** ਏਜੰਟ ਜੋੜਦੇ ਹਾਂ ਜੋ ਯਾਤਰੀ ਦੇ ਬਜਟ ਦੇ ਮੁਕਾਬਲੇ ਯੋਜਨਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ, ਉਹਨਾਂ ਚੀਜ਼ਾਂ ਨੂੰ ਨਿਸ਼ਾਨਾ ਲਗਾਉਂਦਾ ਹੈ ਜੋ ਖ਼ਰਚੇ ਸੀਮਾ ਤੋਂ ਵੱਧ ਚਲਾ ਸਕਦੀਆਂ ਹਨ, ਅਤੇ ਪੈਸਾ ਬਚਾਉਣ ਵਾਲੇ ਵਿਕਲਪ ਸੁਝਾਅ ਦਿੰਦਾ ਹੈ। ਵਰਕਫਲੋ ਹੁਣ ਤਿੰਨ ਏਜੰਟਾਂ ਨੂੰ ਚੱਲਦਾ ਹੈ: 

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ:

1. **ਮਹਿਰਤਯਾਪੂਰਨ ਏਜੰਟ ਬਣਾਓ** — ਹਰ ਇੱਕ ਦਾ ਇੱਕ ਕੇਂਦਰਿਤ ਰੋਲ (ਯੋਜਨਾ ਬਣਾਉਣਾ, ਸਭੰਧਿਤ ਸਹਾਇਤਾ, ਬਜਟ ਸਮੀਖਿਆ)।
2. **ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜੋ** `WorkflowBuilder` ਅਤੇ `add_edge` ਦੀ ਵਰਤੋਂ ਕਰਕੇ।
3. **ਮਲਟੀ-ਏਜੰਟ ਪਾਈਪਲਾਈਨ ਤੋਂ ਨਤੀਜੇ ਸ੍ਰੋਤਬੱਧ ਕਰੋ**, ਇਹ ਟਰੈਕ ਕਰਦੇ ਹੋਏ ਕਿ ਕਿਹੜਾ ਏਜੰਟ ਬੋਲ ਰਿਹਾ ਹੈ।
4. **ਵਰਕਫਲੋ ਨੂੰ ਵਿਸਥਾਰ ਦਿਓ** ਮੌਜੂਦਾ ਏਜੰਟਾਂ ਨੂੰ ਬਦਲੇ ਬਿਨਾਂ ਨਵੇਂ ਏਜੰਟ ਚੇਨ ਵਿੱਚ ਸ਼ਾਮਿਲ ਕਰਕੇ।

ਮਲਟੀ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਹਰ ਏਜੰਟ ਨੂੰ ਸਧਾਰਣ ਰੱਖਦਾ ਹੈ ਜਦਕਿ ਇਹ ਇਕੱਲੇ ਕਿਸੇ ਹੇਠਾਂ ਤੋਂ ਵੱਧ ਉਦਾਹਰਨ, ਹੋਰ ਵਿਸਥਾਰ ਨਾਲ ਸਮੀਖਿਆ ਕੀਤੇ ਨਤੀਜੇ ਪੈਦਾ ਕਰਦਾ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋक्ति**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਯਾਦ ਰੱਖੋ ਕਿ ਆਟੋਮੈਟੇਡ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਤੀਰਤਾ ਹੋ ਸਕਦੀ ਹੈ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ ਪ੍ਰੋਫੈਸ਼ਨਲ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਇਸਤੇਮਾਲ ਕਾਰਨ ਉੱਭਰ ਰਹੀਆਂ ਕੋਈ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਭ੍ਰਮਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
